# Major Montana landowners naive analysis

Crunching data from Montana Cadastral to ID the state's biggest private landowners by acreage, assuming each unique `OwnerName` is a unique owner. That is a bad assumption but produced a useful first-pass analysis.

In [52]:
# Library import and config
import json
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_rows', 100) # Avoids truncation of long data table results

In [ ]:
# Read in data (slow)
# Parcel data from **March 9** download -- may need to update file path
parcels = gpd.read_file('./raw/MontanaCadastral_SHP_3-9-2026/Montana_Cadastral/OWNERPARCEL.shp').to_crs(epsg=4326)

In [59]:
# Group parcels by owner name, count the number in each group and sum by acreage in each group
# Running more than top 10 here because most of the biggest are public entities, which we need to filter out
top_100_landowners = parcels.groupby(['OwnerName']).agg({
    'PARCELID': 'count',
    'TotalAcres': 'sum',
}).sort_values('TotalAcres', ascending=False).head(100)

top_100_landowners.head(30) # Show top 10 to check output

,PARCELID,TotalAcres
OwnerName,,
USDA FOREST SERVICE,16655,8773957.251
USDI BUREAU OF LAND MANAGEMENT,18088,5600054.943
STATE OF MONTANA,12226,4265636.327
UNITED STATES OF AMERICA,5663,1923608.752
USA,3985,1828458.927
USA IN TRUST FOR CROW TRIBE,3836,1302067.777
USA IN TRUST,5279,1139955.995
USA - FOREST SERVICE,2161,1115836.824
BUREAU OF LAND MANAGEMENT,2736,732086.036


In [58]:
# Filter out manually identified public landowners

with open('./known-public-landowners.json', 'r') as file:
    PUBLIC_LANDOWNERS = json.load(file)

top_100_private_only = top_100_landowners[~top_100_landowners.index.isin(PUBLIC_LANDOWNERS)]

# Add rank for display purposes
ranking = top_100_private_only.copy().reset_index()
ranking['Rank'] = ranking.index+1
ranking.rename(columns={'PARCELID': 'NumParcels'}, inplace=True)

# show top 20 for reality check
ranking.head(20) 

,OwnerName,NumParcels,TotalAcres,Rank
0,FARMLAND RESERVE INC,628,301772.012,1
1,GREEN DIAMOND RESOURCE COMPANY,644,291802.938,2
2,WILKS RANCH MONTANA LTD,690,271093.178,3
3,71 RANCH LP,616,249140.977,4
4,SUNLIGHT RANCH COMPANY,706,235991.223,5
5,COFFEE NEFSY LIMITED PARTNERSHIP,389,184258.488,6
6,AMERICAN PRAIRIE FOUNDATION,587,141873.384,7
7,BROKEN O LAND & LIVESTOCK LLC,454,119111.324,8
8,STIMSON LUMBER CO,325,118740.559,9
9,HILLENBRAND JOHN,285,117724.130,10


In [60]:
# Write naive top 10 list to outputs file
ranking.to_json('./outputs/naive-top-10-list.json', orient='records', indent=4, index=False)
